In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from collections import Counter
from itertools import combinations
import seaborn as sns
warnings.filterwarnings("ignore")

In [2]:
# Load data
df = pd.read_csv("train.csv")

In [3]:
len(df)

306226

In [4]:
df.columns

Index(['capturedAt', 'shopId', 'itemId', 'modelId', 'price',
       'priceBeforeDiscount', 'promotionId', 'cat_id', 'stock', 'normal_stock',
       'raw_discount', 'show_discount', 'brand', 'is_free_shipping',
       'is_pre_order', 'item_price_min', 'item_price_max', 'review_rating',
       'total_rating_count', 'cmt_count', 'shop_rating', 'shop_response_rate',
       'shop_follower_count', 'is_official_shop', 'is_verified',
       'is_preferred_plus_seller'],
      dtype='object')

In [5]:
df.describe()

,shopId,itemId,modelId,price,priceBeforeDiscount,promotionId,cat_id,stock,normal_stock,raw_discount,show_discount,item_price_min,item_price_max,review_rating,total_rating_count,cmt_count,shop_rating,shop_response_rate,shop_follower_count
count,3.062260e+05,3.062260e+05,3.062260e+05,3.062260e+05,3.062260e+05,3.062260e+05,306226.000000,3738.0,3738.0,306226.000000,306226.000000,3.062260e+05,3.062260e+05,306226.000000,306226.000000,306226.000000,306226.000000,305027.000000,306226.000000
mean,6.476090e+08,1.938585e+10,1.651984e+11,5.234146e+07,1.885758e+07,1.958224e+14,100432.188472,0.0,0.0,20.499598,20.499598,4.536168e+07,5.965785e+07,4.657069,92.299609,87.292986,4.948080,78.112954,4202.578909
std,4.658536e+08,8.466663e+09,7.341994e+10,9.137196e+07,5.101101e+07,2.605581e+14,291.360308,0.0,0.0,28.531341,28.531341,8.420521e+07,9.787515e+07,1.178862,319.878155,311.354822,0.038938,16.476535,13901.193286
min,1.000692e+06,6.632660e+07,5.246531e+07,1.000000e+05,0.000000e+00,0.000000e+00,100001.000000,0.0,0.0,0.000000,0.000000,1.000000e+05,1.000000e+05,0.000000,0.000000,0.000000,4.500000,9.000000,0.000000
25%,1.002919e+08,1.693105e+10,1.185036e+11,9.900000e+06,0.000000e+00,0.000000e+00,100015.000000,0.0,0.0,0.000000,0.000000,7.900000e+06,1.200000e+07,4.937500,4.000000,4.000000,4.938632,66.000000,216.000000
50%,1.002937e+09,2.247004e+10,1.853718e+11,2.050000e+07,0.000000e+00,0.000000e+00,100631.000000,0.0,0.0,0.000000,0.000000,1.650000e+07,2.590000e+07,4.985000,16.000000,15.000000,4.956926,82.000000,882.000000
75%,1.006572e+09,2.496184e+10,2.239639e+11,5.900000e+07,1.990000e+07,5.352425e+14,100636.000000,0.0,0.0,44.000000,44.000000,5.000000e+07,6.690000e+07,5.000000,64.000000,57.000000,4.972853,92.000000,3558.000000
max,1.206969e+09,2.996089e+10,2.578970e+11,1.660000e+09,1.299000e+09,6.135842e+14,100644.000000,0.0,0.0,88.000000,88.000000,1.660000e+09,1.660000e+09,5.000000,15462.000000,15309.000000,5.000000,100.000000,206855.000000


In [6]:
len(df.columns)

26

In [7]:
df.dtypes

capturedAt                   object
shopId                        int64
itemId                        int64
modelId                       int64
price                         int64
priceBeforeDiscount           int64
promotionId                   int64
cat_id                        int64
stock                       float64
normal_stock                float64
raw_discount                  int64
show_discount                 int64
brand                        object
is_free_shipping             object
is_pre_order                 object
item_price_min                int64
item_price_max                int64
review_rating               float64
total_rating_count            int64
cmt_count                     int64
shop_rating                 float64
shop_response_rate          float64
shop_follower_count           int64
is_official_shop             object
is_verified                  object
is_preferred_plus_seller     object
dtype: object

In [8]:
# Null Values
null_summary = (
    df.isnull()
      .sum()
      .loc[lambda x: x > 0]
      .rename("null_count")
      .to_frame()
)

null_summary["null_pct"] = (
    null_summary["null_count"] / len(df) * 100
).round(2)

null_summary

,null_count,null_pct
stock,302488,98.78
normal_stock,302488,98.78
brand,105664,34.51
shop_response_rate,1199,0.39


In [9]:
df["normal_stock"].value_counts(dropna=False)

normal_stock
NaN    302488
0.0      3738
Name: count, dtype: int64

In [10]:
df["stock"].value_counts(dropna=False)

stock
NaN    302488
0.0      3738
Name: count, dtype: int64

In [11]:
# Duplicates
df.duplicated().sum()

np.int64(2244)

In [12]:
# Constant Data
df["is_preferred_plus_seller"].value_counts()

is_preferred_plus_seller
f    306226
Name: count, dtype: int64

In [13]:
# Check whether duplicate rows still exist after:
# 1. removing exact duplicates,
# 2. dropping stock and normal_stock (mostly missing),
# 3. dropping is_preferred_plus_seller (constant value).
df_dedup_check = df.copy()
df_dedup_check = df_dedup_check[~df_dedup_check.duplicated(keep="first")]
df_dedup_check = df_dedup_check.drop(
    columns=["stock", "normal_stock", "is_preferred_plus_seller"]
)
print(f"Duplicate rows: {df_dedup_check.duplicated().sum()}")
print(f"New length: {len(df_dedup_check)}")

Duplicate rows: 0
New length: 303982
